# Домашнее задание 3 (2026)
## RAG для проверки научных утверждений

**Дедлайн:** будет объявлен преподавателем.  
**Датасет:** [SciFact](https://github.com/allenai/scifact): научные утверждения, корпус из 5 183 аннотаций и размеченные evidence documents/sentences.

Необходимо построить RAG на Hugging Face и FAISS, который находит доказательства и формирует проверяемый ответ со ссылками.

## Правила

1. Работа выполняется в группе до 4 человек; укажите вклад каждого участника.
2. Работа сдаётся в виде одного файла Jupyter Notebook (`.ipynb`) на основе выданного шаблона.
3. Перед сдачей перезапустите ядро и выполните все ячейки по порядку сверху вниз. Убедитесь, что код выполняется без ошибок, и сохраните выполненный Jupyter Notebook. Все выведенные результаты, включая текст, таблицы, графики и метрики, должны быть сохранены; не очищайте выводы ячеек перед сдачей.
4. Отчёт должен содержать код, пояснения, результаты и анализ ошибок.
5. Используйте `claims_train` для разработки, а `claims_dev` — только для финальной оценки. Gold-разметка `claims_test` не опубликована.
6. Сохраняйте `doc_id` и номера предложений на всех этапах.
7. Зафиксируйте seed, embedding-модель, генератор, prompt, параметры декодирования и вычислительную среду.
8. Использование генеративного ИИ для выполнения заданий, написания кода, анализа результатов или подготовки текста отчёта запрещено. При обнаружении нарушения работа оценивается в 0 баллов. Плагиат также ведёт к обнулению; за каждую начатую неделю просрочки снимается 1 балл.

In [ ]:
# Официальный архив данных
# !wget -q -O scifact.tar.gz https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz
# !tar -xzf scifact.tar.gz
SEED = 42

## Часть 1. Данные и разбиение на chunks [2 балла]

В архиве SciFact файл `corpus.jsonl` содержит аннотации научных статей, а `claims_train.jsonl` и `claims_dev.jsonl` — утверждения для обучения и финальной оценки. Ниже corpus означает содержимое `corpus.jsonl`, а train и dev — содержимое соответствующих файлов с claims.

1. Загрузите corpus, train и dev; проверьте идентификаторы и распределение меток. Разбивать на chunks нужно только аннотации из corpus; claims из train и dev используются как запросы и не разбиваются.
2. Самостоятельно напишите функцию, которая делит одну аннотацию из corpus на текстовые фрагменты из последовательных предложений. Функцию можно назвать `make_chunks`. Параметр `chunk_size` задаёт число предложений в одном фрагменте, а `overlap` — число предложений, повторяющихся в соседних фрагментах. Для каждого фрагмента сохраняйте текст, `chunk_id`, `doc_id` и номера исходных предложений.
3. Сравните разные значения `chunk_size`, а также варианты с overlap и без него. Сообщите число полученных chunks и распределение их длины.
4. Объясните, как overlap помогает не разрывать доказательства на границах chunks, и выберите подходящую конфигурацию для последующего поиска.

In [ ]:
def make_chunks(sentences, doc_id, chunk_size, overlap):
    """
    Разделить список предложений одной аннотации на текстовые фрагменты.

    Возвращаемое значение: список словарей с полями
    text, chunk_id, doc_id и sentence_ids.
    """
    # TODO: реализуйте разбиение на chunks
    pass

## Часть 2. Текстовые эмбеддинги и поиск в FAISS [2.5 балла]

Эмбеддинг — это числовой вектор, представляющий смысл текста. FAISS хранит векторы chunks и по вектору claim находит наиболее близкие фрагменты.

1. Выберите модель из библиотеки sentence-transformers. С её помощью преобразуйте текст каждого chunk в эмбеддинг, сообщите идентификатор модели и размерность вектора. Нормализуйте эмбеддинги до единичной длины и объясните цель нормализации.
2. Постройте FAISS-индекс из эмбеддингов chunks и сохраните соответствие между позицией вектора и метаданными chunk. При поиске по L2 сравнивается евклидово расстояние: чем оно меньше, тем ближе тексты. При поиске по inner product сравнивается скалярное произведение: чем оно больше, тем ближе тексты; для нормализованных векторов это эквивалентно сравнению cosine similarity. Выберите вариант и обоснуйте решение.
3. Самостоятельно напишите функцию `search(claim, k)`. Здесь `claim` — текст научного утверждения, а `k` — число результатов, возвращаемых поиском. Функция должна преобразовать claim в эмбеддинг тем же способом, выполнить поиск в FAISS и вернуть найденные фрагменты. Для каждого результата верните `score`, то есть расстояние или значение сходства из FAISS, а также `text`, `chunk_id`, `doc_id` и `sentence_ids`.
4. Для репрезентативных claims покажите найденные результаты и вручную объясните удачные и неудачные случаи.

In [ ]:
def search(claim, k):
    """
    Найти в FAISS фрагменты, наиболее близкие к тексту claim.

    Возвращаемое значение: список словарей с полями
    score, text, chunk_id, doc_id и sentence_ids.
    """
    # TODO: закодируйте claim, выполните поиск и верните результаты
    pass

## Часть 3. Эксперименты с поиском [1.5 балла]

В разметке SciFact gold `doc_id` — это идентификаторы документов, которые действительно содержат доказательства для claim. Для каждого claim с доказательствами Recall@k равен доле gold-документов, попавших в первые `k` результатов поиска; итоговая метрика усредняется по таким claims. Claims без размеченных доказательств при расчёте этой метрики не учитываются.

1. Вычислите Recall@1/3/5 для выбранных конфигураций chunking.
2. Сравните разные значения `k` и варианты `chunk_size`/`overlap`. Поясните, как параметры влияют на качество поиска.
3. Выберите параметры только по train.
4. На репрезентативных примерах разберите ошибки поиска и объясните влияние терминологии, длины и границ chunks.

## Часть 4. Генерация RAG-ответов [3 балла]

1. Выберите открытую генеративную модель, которую можно запустить в доступной вычислительной среде, и сообщите её идентификатор.
2. Самостоятельно напишите `build_prompt(claim, context_chunks)`. Функция объединяет проверяемое утверждение и найденные доказательства в текст инструкции для генеративной модели. В prompt потребуйте использовать только переданные доказательства и соблюдать формат ответа.
3. Самостоятельно напишите `generate_answer(prompt)`. Функция передаёт prompt выбранной модели и возвращает сгенерированный текст.
4. Самостоятельно напишите `rag(claim, k)`. Функция должна последовательно вызвать `search`, `build_prompt` и `generate_answer`; параметр `k` определяет число извлекаемых из FAISS фрагментов.
5. Ответ должен содержать подходящий вердикт: `SUPPORT`, если доказательства подтверждают claim; `CONTRADICT`, если они ему противоречат; `NOT ENOUGH INFO`, если доказательств недостаточно. После вердикта приведите краткое объяснение и ссылки вида `[doc_id:sentence_id]`.
6. Для одинаковых claims сравните три условия: без контекста — модель получает только claim; со случайными chunks — модель получает случайные фрагменты corpus; с найденными chunks — модель получает результаты `search`. Объясните, улучшает ли поиск ответы.
7. Сравните разные значения `k` и варианты prompt на одинаковых claims; оцените правильность вердикта и ссылок.

In [ ]:
def build_prompt(claim, context_chunks):
    """Собрать prompt из claim и найденных фрагментов доказательств."""
    # TODO: реализуйте шаблон prompt
    pass

def generate_answer(prompt):
    """Передать prompt генеративной модели и вернуть её ответ."""
    # TODO: реализуйте генерацию
    pass

def rag(claim, k):
    """Выполнить поиск доказательств и сформировать итоговый ответ."""
    context_chunks = search(claim, k)
    prompt = build_prompt(claim, context_chunks)
    return generate_answer(prompt)

## Часть 5. Финальная оценка и итоги [1 балл]

На финальном этапе оцените лучшую систему на `claims_dev`. Сообщите macro $F_1$ для вердиктов. Корректной считается ссылка, в которой одновременно совпадают gold `doc_id` и номер gold-предложения; для claims с размеченными доказательствами сообщите долю ответов, содержащих хотя бы одну такую ссылку. На репрезентативных примерах разделите ошибки на ошибки поиска и ошибки генерации, затем сформулируйте основные выводы.